# 02 Shared-Routed Head MoE 12h

FedSTO reproduction scheduleを保ったまま、DQA x MOEをhead中心のshared/routed expertとして入れる実験。server repaired modelをshared expert、client updatesをweather/client/class/quality routed expertとして扱い、source/cloudy trust-regionで安全な候補だけを次roundへ渡す。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from datetime import datetime

REPO_ROOT = Path('/app/Object_Detection')
EXP_ROOT = REPO_ROOT / 'dynamic_quality_aware_classwise_aggregation' / 'dqa_moe_trust_region'
RUNNER = EXP_ROOT / 'scripts' / 'run_dqa_moe_trust_region.py'
WORKSPACE = EXP_ROOT / 'output' / '02_shared_routed_head_moe_12h'
LOG_DIR = EXP_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
WORKSPACE.mkdir(parents=True, exist_ok=True)

print('runner:', RUNNER)
print('workspace:', WORKSPACE)

## Design

- FedSTO側は前回と同じ: warmup=50, phase1=20, phase2=20, local/server epoch=1。
- phase2 round 6以降だけDQAを入れる。
- client local EMAが生成したpseudo label statsを保存し、class別count/quality/balanceをrouterに使う。
- YOLO headの最終conv channelはclassごとにexpert weightを変える。box/objectness channelはglobal weather routingを使う。
- 候補は `head` / `head_bn` と `balanced` / `rainy` / `snowy` routing focus の6個。
- source/cloudyで悪化する候補は採用しない。

In [ ]:
CMD = [
    sys.executable, str(RUNNER),
    '--workspace-root', str(WORKSPACE),
    '--warmup-epochs', '50',
    '--phase1-rounds', '20',
    '--phase2-rounds', '20',
    '--batch-size', '128',
    '--workers', '32',
    '--gpus', '2',
    '--master-port', '29543',
    '--dqa-start-round', '6',
    '--dqa-router-mode', 'shared_routed',
    '--dqa-router-candidates', 'balanced,rainy,snowy',
    '--dqa-candidate-scopes', 'head,head_bn',
    '--dqa-candidate-lambda-multipliers', '0.60',
    '--dqa-max-candidates', '6',
    '--dqa-lambda-start', '0.012',
    '--dqa-lambda-end', '0.045',
    '--dqa-max-relative-update', '0.008',
    '--dqa-acceptance-tolerance-map50', '0.003',
    '--dqa-acceptance-tolerance-map50-95', '0.002',
    '--dqa-score-map50-weight', '0.20',
    '--dqa-score-recall-weight', '0.05',
    '--dqa-router-proxy-weight', '0.001',
    '--dqa-collect-pseudo-stats',
    '--dqa-pseudo-quality-mode', 'feature_balanced',
    '--dqa-classwise-head-routing',
    '--val-batch-size', '32',
    '--run-final-eval',
    '--discord',
]

print(' '.join(CMD))

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = LOG_DIR / f'02_shared_routed_head_moe_12h_{timestamp}.log'
print('log:', log_path)

with log_path.open('w', encoding='utf-8', buffering=1) as log:
    log.write(' '.join(CMD) + '\n')
    process = subprocess.Popen(
        CMD,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy(),
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f'run failed with code {return_code}; see {log_path}')
print('done:', WORKSPACE)

In [ ]:
import pandas as pd

summary_csv = WORKSPACE / 'dqa_moe_round_summary.csv'
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    display(df.tail(20))
else:
    print('missing:', summary_csv)

report = WORKSPACE / 'validation_reports' / 'paper_protocol_eval_summary.md'
if report.exists():
    print(report.read_text(encoding='utf-8'))
else:
    print('missing:', report)